# Definición de la caja de docking.

En autoDock Vina, la simulación de docking no ocurre en todo el espacio del universo. Para ahorrar tiempo de cómputo y ser precisos, definimos una Grid Box (cjada de rejilla). Esta caja está definida por seis parámetros esenciales.
1. Centro (x,y,z): Las coordenadas tridimencionales cartesianas del punto del bolsillo activo.
2. Tamaño (x,y,z): Las dimensiones de la caja en Angstroms que deben ser suficientemente grandes para permitir que el lingando rote y explore conformaciones, pero no tan grandes como para perder especificidad.

Como ya hemos extraído los ligandos de control que están co-cristalizados exactamente en els itio activo de interés, tenemos la clave biológica. El centro geométrico de estos ligandos representa el centro perfecto para nuestra caja de docking de PBP2 y GyrB.

## Desafío matemático y algorítimico.

Para encontrar el centro perfecto de nuestra caja, necesitamos calcular el promedio de las coordenadas cartesianas (x, y ,z) de todos los átomos pesados del lingando de control.
La fórmula del centro geométric:
  $$C_x = \frac{1}{N} \sum_{i=1}^{N} x_i, \quad C_y = \frac{1}{N} \sum_{i=1}^{N} y_i, \quad C_z = \frac{1}{N} \sum_{i=1}^{N} z_i$$

  Donde:
  - $N$ es el número total de átomos del ligando.
  - $x_i, y_i, z_i$ son las coordenadas del átomo $i$ en el archivo PDB.

In [58]:
# calcular_centro.py

import os
import numpy as np

def find_coordinates(path_file):

    if not os.path.exists(path_file):
        print(f"Error: El archivo {path_file} no existe")
        return None
    
    atomes_coordinates = []

    with open(path_file, "r") as file:
        for line in file.read().split("\n"):
            # Evaluar ATOM o HETATM
            if line.startswith("HETATM") or line.startswith("ATOM"):
                # Extraer el rango extacto de 8 caracteres opr coordenada
                x = line[30:38].strip()
                y = line[38:46].strip()
                z = line[46:54].strip()

                atomes_coordinates.append([float(x), float(y), float(z)])
            else:
                continue
    
    if not atomes_coordinates:
        print("No se encontraron coordenadas en el archivo.")
    
    #Convertimos a matrix NumPy
    coordinates_matrix = np.array(atomes_coordinates)
    return coordinates_matrix.mean(axis=0)

def generar_config_vina(receptor_nombre, centro, tamaño_caja, archivo_salida):
        """
        Genera automáticamente el archivo de configuración conf.txt para AutoDock Vina.
        """
        carpeta_salida = "../docking/vina_input"
        os.makedirs(carpeta_salida, exist_ok=True)

        ruta_completa = os.path.join(carpeta_salida, archivo_salida)

        # Si el centro no se calculó correctamente
        if centro is None:
            print(f"Error: No se pudo generar la configuración para {receptor_nombre} porque el centro es None.")
            return

        try:
            with open(ruta_completa, "w") as f:
                f.write(f"# Configuración de AutoDock Vina para {receptor_nombre}\n")
                f.write("# Generado automáticamente por Andrés Pérez Palacios\n\n")

                # Rutas de entrada (se asumen relativas al ejecutar Vina en la raíz)
                f.write(f"receptor       = data/receptors/{receptor_nombre}_clean.pdbqt\n")
                # Dejamos el ligando como parámetro que se puede pasar por consola o definir genérico
                f.write(f"ligand         = data/ligands/cambiar_por_ligando.pdbqt\n\n")

                # Centro de la Caja (Grid Box Center)
                f.write(f"center_x       = {centro[0]:.3f}\n")
                f.write(f"center_y       = {centro[1]:.3f}\n")
                f.write(f"center_z       = {centro[2]:.3f}\n\n")

                # Tamaño de la Caja (Grid Box Size)
                f.write(f"size_x         = {tamaño_caja:.1f}\n")
                f.write(f"size_y         = {tamaño_caja:.1f}\n")
                f.write(f"size_z         = {tamaño_caja:.1f}\n\n")

                # Parámetros fijos de Vina (del context.md)
                f.write("exhaustiveness = 32\n")
                f.write("num_modes      = 9\n")
                f.write("energy_range   = 3\n")

            print(f"¡Configuración guardada con éxito en: {ruta_completa}!")

        except Exception as e:
            print(f"Error al escribir el archivo de configuración: {e}")

    
# Ruta de los ligandos
ligand_1 = "../data/ligands/Ceftaroline_3D.pdb"
ligand_2 = "../data/ligands/07N_GyrB_3D.pdb"            # No sé si este es el mismo ligando (?) o son ligandos de diferentes dianas

# Cálculo de centros
centro_pbp2a = find_coordinates(ligand_1)
centro_gyrb = find_coordinates(ligand_2)

print(f"Centro PBP2a (3ZG0) -> X: {centro_pbp2a[0]:.3f}, Y: {centro_pbp2a[1]:.3f}, Z: {centro_pbp2a[2]:.3f}")
print(f"Centro GyrB  (3TTZ) -> X: {centro_gyrb[0]:.3f}, Y: {centro_gyrb[1]:.3f}, Z: {centro_gyrb[2]:.3f}")


# --- EJECUCIÓN ---
# Generamos los conf.txt de las dos dianas que ya calculamos
generar_config_vina("PBP2a_3ZG0", centro_pbp2a, 22.0, "conf_pbp2a.txt")
generar_config_vina("GyrB_3TTZ", centro_gyrb, 22.0, "conf_gyrb.txt")

Centro PBP2a (3ZG0) -> X: -4.043, Y: 37.118, Z: 77.570
Centro GyrB  (3TTZ) -> X: 8.203, Y: -8.019, Z: 15.475
¡Configuración guardada con éxito en: ../docking/vina_input/conf_pbp2a.txt!
¡Configuración guardada con éxito en: ../docking/vina_input/conf_gyrb.txt!


 ## 🧪 El Desafío de MurG: ¿Cómo definir su sitio activo sin ligando co-cristalizado?

  MurG es una diana sumamente interesante. Al ser un modelo de AlphaFold ( MurG_AF-Q6GGZ0-F1.pdb ), no tiene ningún ligando co-
  cristalizado experimental dentro del archivo. ¿Cómo resolvemos esto en el mundo real?

  Aquí tienes las dos estrategias estándar en bioinformática estructural, y ambas son excelentes ideas:

  ### Estrategia A: Superposición Estructural (Alineamiento con Moldes)

  En la literatura sabemos que MurG de E. coli tiene estructuras en el PDB que sí tienen co-cristalizado su sustrato natural o
  inhibidores (como el PDB  1F0K ).

  1. Haces un alineamiento estructural (comando  align  en PyMOL) entre tu modelo AlphaFold de S. aureus y la estructura de E.
  coli  1F0K .
  2. Al superponerse las proteínas casi perfectamente, el ligando del molde  1F0K  quedará ubicado exactamente en el "bolsillo
  virtual" de tu modelo AlphaFold.
  3. Calculas el centro de ese ligando alineado y ¡listo!, tienes tu caja.

  ### Estrategia B: Docking Ciego y Detección de Cavidades (Tu propuesta de AlphaFold / CB-Dock2)

  ¡Tu idea es excelente y muy moderna! Correr un docking ciego en servidores académicos como CB-Dock2 (gratuito) o usar
  herramientas de detección de cavidades es ideal.

  • CB-Dock2 utiliza un algoritmo llamado Curvature-based Cavity Detection. Escanea toda la superficie de la proteína MurG,
  identifica dónde está el bolsillo más grande y profundo (que casi siempre es el sitio activo donde se une el donador de azúcar
  UDP-GlcNAc) y ejecuta AutoDock Vina ahí de forma automática, dándote las coordenadas exactas de la caja.
  ──────

In [55]:
import os
import urllib.request


def descargar_3d_pubchem(cid, output):

    # Definimos la carpeta de destino:
    folder = "../data/ligands"
    os.makedirs(folder, exist_ok=True)
    full_path = os.path.join(folder,f"{output}_3D.sdf")

    # Construimos la url de Pubchem
    URL =  f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/SDF?record_type=3d"
    print(f"Intentando descargar {output} (CID {cid}) desde PubChem...")
    try:
        #1. Abrimos la conexión web
        with urllib.request.urlopen(URL) as response:
            data = response.read()

        with open(full_path, "wb") as file: # "wb" significa escribir en modo binario
            file.write(data)
        
        print("Descarga completa con éxito!")
    except urllib.error.HTTPError as e:
        print(f"Error HTTP en la descarga {e.code}")
    except Exception as e:
        print(f"Ocurrió un error inesperado {e}")


# Ejecución
OUTPUT = "Quercetin"
CID = 5280343
descargar_3d_pubchem(cid= CID, output= OUTPUT)

Intentando descargar Quercetin (CID 5280343) desde PubChem...
Descarga completa con éxito!
